# Day 1 Project — Solution: Personal AI Briefing Generator

> You should only be reading this after completing your own version.
> Compare your approach — there is no single correct solution.

---

In [ ]:
import asyncio
import os
from datetime import date

import edge_tts

VOICE = "en-US-JennyNeural"

HEADLINES = [
    "OpenAI releases new reasoning model with improved code generation",
    "Google DeepMind's AlphaFold 3 predicts protein interactions with record accuracy",
    "Anthropic raises funding to accelerate Constitutional AI research",
    "Microsoft integrates AI coding assistant into Visual Studio Code natively",
    "South African startup uses computer vision to detect crop disease in real time",
]


def build_script(headlines: list[str]) -> str:
    """
    Format raw headlines into a spoken briefing script.

    Key decisions:
    - Add intro and outro to frame the content
    - Use transition phrases so it doesn't sound like a list
    - Include today's date for context
    - Add a short pause signal between items using commas and full stops
    """
    today = date.today().strftime("%A, %d %B %Y")

    transitions = [
        "In our first story today,",
        "Next,",
        "In other news,",
        "Also making headlines,",
        "And finally,",
    ]

    intro = (
        f"Good morning. This is your AI briefing for {today}. "
        f"Here are the five stories you need to know."
    )

    body_parts = []
    for i, headline in enumerate(headlines):
        transition = transitions[i] if i < len(transitions) else "Additionally,"
        body_parts.append(f"{transition} {headline}.")

    outro = (
        "That's your AI briefing for today. "
        "Stay curious, keep building, and I'll see you tomorrow."
    )

    return "  ".join([intro] + body_parts + [outro])


async def generate_briefing(script: str, output_path: str) -> None:
    """Generate audio from script and save to output_path."""
    communicate = edge_tts.Communicate(script, VOICE)
    await communicate.save(output_path)


async def main() -> None:
    # Build script
    script = build_script(HEADLINES)

    # Timestamped filename
    today_str = date.today().strftime("%Y%m%d")
    output_path = f"briefing_{today_str}.mp3"

    # Generate audio
    print(f"Generating briefing...")
    await generate_briefing(script, output_path)

    # Summary
    size_kb = os.path.getsize(output_path) / 1024
    # Rough estimate: edge_tts outputs ~24kbps, so bytes / 3000 ≈ seconds
    est_seconds = os.path.getsize(output_path) / 3000

    print(f"File     : {output_path}")
    print(f"Size     : {size_kb:.1f} KB")
    print(f"Duration : ~{est_seconds:.0f} seconds")


await main()

---
## What to Compare

**Script construction (`build_script`)**  
Did you write transition phrases, or just join the headlines? A good spoken script sounds different from a bullet list. Natural transitions are what separate a robotic read from a real briefing.

**Async structure**  
Did you use `async def` + `asyncio.run()`? Or did you try calling `.save()` synchronously? Edge TTS is async — understanding why (network I/O) is important.

**Filename**  
Did you use `date.today()` for the timestamp? A common mistake is hardcoding dates or using `datetime.now()` when only the date is needed.

**Duration estimate**  
Did you estimate listening time? This is a useful real-world output that tells the user something meaningful about the file — not just the size.

**What could be improved in this solution:**  
- The duration estimate is rough (bytes / 3000). A more accurate method uses `ffprobe` to read the actual audio metadata — you'll build that later in the course.
- Headlines are hardcoded. In a real system, these would come from an RSS feed or a news API — you'll do that in Section 2 (Automation).
- There's no error handling for the TTS call. What happens if you're offline? That's worth thinking about.